**Mount Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Import Libraries**

In [2]:
import os
import joblib
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model

**Load Trained Model + Scaler + Data**

In [7]:
BASE_DIR = "/content/drive/MyDrive/c01-price-forecasting/"

MODEL_PATH = BASE_DIR + "models/lstm_model.h5"
SCALER_PATH = BASE_DIR + "data/model_inputs/lstm/scaler.pkl"
DATA_PATH = BASE_DIR + "data/processed/cleaned_data.csv"

model = load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print("LSTM model + scaler and data loaded")

LSTM model + scaler and data loaded


**Create Feature Function**

In [8]:
def create_features(df, district, input_date):

    input_date = pd.to_datetime(input_date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < input_date]

    if len(df_d) < 12:
        raise ValueError("Not enough historical data")

    df_last = df_d.tail(12)

    last_row = df_last.iloc[-1]

    features = {}

    # ===== TIME FEATURES =====
    features['year'] = input_date.year
    features['month'] = input_date.month
    features['week'] = input_date.isocalendar().week
    features['week_of_year'] = input_date.isocalendar().week

    # Sin/Cos encoding
    features['week_sin'] = np.sin(2 * np.pi * features['week'] / 52)
    features['week_cos'] = np.cos(2 * np.pi * features['week'] / 52)

    # ===== MARKET FEATURES =====
    features['min_price'] = last_row['min_price']
    features['max_price'] = last_row['max_price']
    features['production_total'] = last_row['production_total']

    features['price_range'] = last_row['max_price'] - last_row['min_price']

    # ===== LAG FEATURES =====
    features['lag_1'] = df_last.iloc[-1]['avg_price']
    features['lag_2'] = df_last.iloc[-2]['avg_price']
    features['lag_4'] = df_last.iloc[-4]['avg_price']
    features['lag_12'] = df_last.iloc[-12]['avg_price']

    # ===== ROLLING FEATURES =====
    features['rolling_mean_4'] = df_last['avg_price'].tail(4).mean()
    features['rolling_std_4'] = df_last['avg_price'].tail(4).std()

    # ===== AVERAGE FEATURES =====
    features['price_4w_avg'] = df_last['avg_price'].tail(4).mean()
    features['price_8w_avg'] = df_last['avg_price'].tail(8).mean()

    # ===== CHANGE FEATURE =====
    features['price_change'] = df_last.iloc[-1]['avg_price'] - df_last.iloc[-4]['avg_price']

    # ===== SEASON =====
    features['season_Yala'] = 0 if input_date.month in [9,10,11,12,1,2,3] else 1

    return features

**Create LSTM Input**

In [9]:
def prepare_lstm_input(df_d, scaler, feature_cols, timesteps=12):

    df_last = df_d.tail(timesteps)

    X = df_last[feature_cols].values

    # Add dummy column for avg_price (for scaler compatibility)
    full = np.zeros((len(X), len(feature_cols) + 1))
    full[:, :-1] = X

    scaled = scaler.transform(full)[:, :-1]

    # Reshape for LSTM
    X_lstm = scaled.reshape(1, timesteps, len(feature_cols))

    return X_lstm

**Prediction Function**

In [10]:
def predict_price_lstm(district, date):

    date = pd.to_datetime(date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < date]

    # Feature columns (same as training)
    feature_cols = list(scaler.feature_names_in_[:-1])

    # Prepare input
    X_lstm = prepare_lstm_input(df_d, scaler, feature_cols)

    # Predict (scaled)
    y_pred_scaled = model.predict(X_lstm)[0][0]

    # Inverse scale
    dummy = np.zeros((1, len(feature_cols) + 1))
    dummy[0, -1] = y_pred_scaled

    y_pred = scaler.inverse_transform(dummy)[0, -1]

    print(f"\n District: {district}")
    print(f" Date: {date}")
    print(f" LSTM Predicted Price: {y_pred:.2f}")

    return float(y_pred)

**Test Section**

In [11]:
predict_price_lstm("Anuradhapura", "2026-05-01")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step

 District: Anuradhapura
 Date: 2026-05-01 00:00:00
 LSTM Predicted Price: 111.49


111.49248830676078

**Forecast Function**

In [12]:
from datetime import timedelta

def forecast_lstm(district, start_date, weeks=4):

    start_date = pd.to_datetime(start_date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < start_date]

    df_future = df_d.copy()
    predictions = []

    feature_cols = list(scaler.feature_names_in_[:-1])

    current_date = start_date

    for i in range(weeks):

        X_lstm = prepare_lstm_input(df_future, scaler, feature_cols)

        y_pred_scaled = model.predict(X_lstm)[0][0]

        # Inverse scaling
        dummy = np.zeros((1, len(feature_cols) + 1))
        dummy[0, -1] = y_pred_scaled
        y_pred = scaler.inverse_transform(dummy)[0, -1]

        predictions.append({
            "date": current_date,
            "predicted_price": float(y_pred)
        })

        # Add prediction to dataset
        new_row = df_future.iloc[-1].copy()
        new_row['date'] = current_date
        new_row['avg_price'] = y_pred

        df_future = pd.concat([df_future, pd.DataFrame([new_row])], ignore_index=True)

        current_date += timedelta(weeks=1)

    return pd.DataFrame(predictions)

**Test Section**

In [14]:
forecast_lstm("Anuradhapura", "2026-05-21", weeks=8)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,date,predicted_price
0,2026-05-21,111.492488
1,2026-05-28,110.359393
2,2026-06-04,109.556709
3,2026-06-11,108.947474
4,2026-06-18,108.602705
5,2026-06-25,108.481192
6,2026-07-02,108.500418
7,2026-07-09,108.654262
